In [ ]:
# conda create --name internvl --clone qwen
# conda activate internvl
# module load cuda/12.2
# pip install lmdeploy>=0.7.3

In [ ]:
import re
import csv
import glob
import json

from lmdeploy import pipeline, TurbomindEngineConfig, ChatTemplateConfig
from lmdeploy.vl import load_image
from tqdm import tqdm


model = 'OpenGVLab/InternVL3-38B-AWQ'
pipe = pipeline(model, backend_config=TurbomindEngineConfig(session_len=16384, tp=1), chat_template_config=ChatTemplateConfig(model_name='internvl2_5'))

In [ ]:
text = """You are an image analysis tool specialized in facial attribute classification. 
          For the provided face image, output a JSON object with the following attributes:
        
            {
              "gender": ["male", "female"],
              "age": ["young", "middle-aged", "senior"],
              "skin_color": ["light", "medium", "dark"],
              "ancestry": ["asian", "south_asian", "black", "latino/hispanic", "middle_eastern", "white", "indigenous"],
              "hair_color": ["black", "brown", "red", "blonde", "gray", "other"],
              "bangs": ["yes", "no"],
              "bald": ["yes", "no"],
              "beard": ["no", "mustache", "stubble", "full"],
              "glasses": ["no", "regular", "sun"],
              "headwear": ["no", "beanie", "cap", "hat", "headband", "hijab", "helmet", "turban"],
            }

          Ensure the labeling is based on visible evidence only. If an attribute is unclear, return "unknown".
          
          Only output the JSON without any additional explanation or text.
          
          Example JSON output:
          
          {
            "gender": "female",
            "age": "middle-aged",
            "skin_color": "light",
            "ancestry": "asian",
            "hair_color": "black",
            "bangs": "no",
            "bald": "no",
            "beard": "no",
            "glasses": "sun",
            "headwear": "beanie",
          }
          """

In [ ]:
default_keys = ["gender", "age", "skin_color", "ancestry", "hair_color", "bangs", 
                "bald", "beard", "glasses", "headwear"]

# Function to process a single image
def process_images(batch_images):
    # Sample messages for batch inference
    prompts = [(text, load_image(img)) for img in batch_images]
    response = pipe(prompts)
    result = []
    for i in range(len(batch_images)):
        # Extracting the JSON part
        json_output = re.search(r"\{.*\}", response[i].text, re.DOTALL).group()
        json_output = re.sub(r',\s*}', '}', json_output)  # This removes a comma just before a closing curly brace
        data = json.loads(json_output)
        result.append([batch_images[i].replace('../../../data/RFW/data/', '')] + [data.get(key, "unknown") for key in default_keys])
    return result

In [ ]:
init = 0
files = glob.glob('../../../data/RFW/data/*/*/*')

import os
os.makedirs("../../../results/annotations", exist_ok=True)
output_file = "../../../results/annotations/internvl_labels.csv"

# Column names
columns = ["File Name", "Gender", "Age", "Skin Color", "Ancestry", "Hair Color", 
           "Bangs", "Bald", "Beard", "Glasses", "Headwear"]

# Chunk settings
chunk_size = 32  # Number of images processed per batch

# Write CSV header
with open(output_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)  # CSV Header

# Process in chunks
for start_idx in tqdm(range(init, len(files), chunk_size), desc="Processing Images"):
    end_idx = min(start_idx + chunk_size, len(files))
    chunk = files[start_idx:end_idx]
    results = process_images(chunk)
    # Save chunk results to CSV
    with open(output_file, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)